# HW4 — Unsupervised Mini-Lab: Factor Structure & Regimes

**FIN 7057 · Due after Week 8 · 100 pts · short assignment**

Fill in the `# TODO`s. A short, interpretive notebook — insight over volume.


## 0. Setup & a multi-asset loader (synthetic fallback)


In [ ]:
# !pip install -q yfinance scikit-learn
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
SEED=0; np.random.seed(SEED)
TICKERS=['XLK','XLF','XLE','XLV','XLY','XLP','XLI','XLU','XLB','XLRE']   # sector ETFs (TODO: your choice)
START, END='2015-01-01','2024-12-31'
def load_panel(tickers, start, end):
    try:
        import yfinance as yf
        df=yf.download(tickers, start=start, end=end, progress=False, auto_adjust=True)['Close']
        rets=np.log(df).diff().dropna()
        if rets.shape[1]>=2: print(f'loaded {rets.shape}'); return rets
        raise RuntimeError('too few')
    except Exception as e:
        print(f'synthetic panel ({e!r})'); rng=np.random.default_rng(SEED)
        d=2000; mkt=rng.standard_normal(d)*0.01; sec=rng.standard_normal((d,3))*0.006
        cols={}
        for i,t in enumerate(tickers): cols[t]=mkt+0.8*sec[:,i%3]+rng.standard_normal(d)*0.01
        return pd.DataFrame(cols, index=pd.bdate_range(start,periods=d))
rets=load_panel(TICKERS,START,END)


## Part A — Panel & correlation (15 pts)


In [ ]:
# TODO: inspect rets (missing values, date range). Then the correlation heatmap:
C=rets.corr()
plt.figure(figsize=(6,5)); plt.imshow(C,vmin=-1,vmax=1,cmap='RdBu_r'); plt.colorbar()
plt.xticks(range(len(C)),C.columns,rotation=90); plt.yticks(range(len(C)),C.columns)
plt.title('Return correlation'); plt.tight_layout(); plt.show()
# TODO (written): comment on overall co-movement.


## Part B — PCA & the market factor (35 pts)


In [ ]:
pca=PCA().fit(rets.values)
evr=pca.explained_variance_ratio_
# TODO: scree plot of evr (bar chart).
print(f'PC1 explains {evr[0]:.1%}')
pc1=rets.values@pca.components_[0]
mkt=rets.mean(axis=1).values                       # equal-weight 'market'
print(f'|corr(PC1, equal-weight market)| = {abs(np.corrcoef(pc1,mkt)[0,1]):.2f}')
print('PC1 loadings all same sign:', bool(np.all(pca.components_[0]>0) or np.all(pca.components_[0]<0)))
# TODO: show PC1 loadings per asset; discuss (Part B2).


## Part C — pick ONE (30 pts)
**Option 1:** interpret PC2/PC3 loadings (do they separate sectors?).  
**Option 2:** regime clustering (k-means on rolling vol/trend of the market proxy).


In [ ]:
UNSUPERVISED_OPTION=None  # TODO: set to 'factors' or 'regimes'
option_result=None        # TODO: store your loadings table or regime summary here
# TODO: implement your chosen option.
# Option 1 hint: pd.DataFrame(pca.components_[1:3].T, index=rets.columns, columns=['PC2','PC3'])
# Option 2 hint:
#   from sklearn.cluster import KMeans
#   m=pd.Series(mkt); z=pd.DataFrame({'vol':m.rolling(21).std().shift(1),
#                                    'mean':m.rolling(21).mean().shift(1)}).dropna()
#   feat=z.to_numpy()
#   lab=KMeans(3,n_init=10,random_state=0).fit_predict(StandardScaler().fit_transform(feat))


## Part D — Interpretation, stability & leakage caveat (20 pts)
TODO (>=200 words): what structure did you find; is it stable across sub-periods; why is a
full-sample regime label not tradeable; why fit PCA on train only for predictive use?


## Extensions (optional)

The required core is everything above, and the self-check tests only that. These extensions are optional and **not** required for full marks — a clean, well-reasoned core beats a sprawling one. If you want to go further:

- Cluster rolling market features (e.g., KMeans) into regimes and interpret them.
- Recompute PCA on rolling windows and report how stable PC1's loadings are.
- Add a second asset panel (a different sector set) and compare the factor structure.

## Self-check (auto-graded)

Run this cell **last**, after the whole notebook. Every line must print **PASS** before you submit — it verifies the mechanical requirements (reproducibility, leak-free features, time-ordered splits, metrics computed). Your written answers and judgment are graded by hand.

In [ ]:
# ================= SELF-CHECK (auto-graded) =================
# These checks verify mechanics, not a preferred economic finding.
def _check(name, fn, hint=""):
    try:
        ok=bool(fn())
    except Exception as e:
        ok,hint=False,hint or f"({type(e).__name__}: {e})"
    print(("PASS  " if ok else "FAIL  ")+name+("" if ok else "   -> "+hint))
    return ok
_r=[]
_r.append(_check("reproducibility: SEED set",lambda:SEED is not None,"set SEED"))
_r.append(_check(">= 10 assets, no NaNs",lambda:rets.shape[1]>=10 and bool(rets.notna().all().all()),"load a clean panel of at least 10 assets"))
_r.append(_check("PCA fit; variance ratios are finite and sum to 1",lambda:len(evr)==rets.shape[1] and np.isfinite(evr).all() and abs(float(np.sum(evr))-1)<1e-6,"fit PCA on the clean panel"))
_r.append(_check("PC1 comparison is computable",lambda:len(pc1)==len(mkt) and np.isfinite(np.corrcoef(pc1,mkt)[0,1]),"compute PC1 scores and the equal-weight comparison"))
_r.append(_check("one unsupervised option completed",lambda:UNSUPERVISED_OPTION in {'factors','regimes'} and option_result is not None,"set UNSUPERVISED_OPTION and store the result"))
print(f"\n{sum(_r)}/{len(_r)} mechanical checks passed.")
assert all(_r),"Fix the FAIL items above before submitting."


## AI-use disclosure & reproducibility
TODO.
